In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
import logging
import os
import re
import sys
import numpy as np
from itertools import chain
from gensim.models import KeyedVectors
import gensim
import pandas as pd
import torch
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
import pickle

# =================== 超参（和你本地保持一致） ===================
embed_size = 300
max_len = 512

# =================== Kaggle路径【自行核对修改】 ===================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
GLOVE_PATH = "/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt"

# =================== 文本清洗函数（原版不动） ===================
def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    return words

def encode_samples(tokenized_samples, word_to_idx):
    features = []
    for sample in tokenized_samples:
        feature = []
        for token in sample:
            if token in word_to_idx:
                feature.append(word_to_idx[token])
            else:
                feature.append(0)
        features.append(feature)
    return features

def pad_samples(features, maxlen=max_len, PAD=0):
    padded_features = []
    for feature in features:
        if len(feature) >= maxlen:
            padded_feature = feature[:maxlen]
        else:
            padded_feature = feature.copy()
            while len(padded_feature) < maxlen:
                padded_feature.append(PAD)
        padded_features.append(padded_feature)
    return padded_features

# =================== 主流程 ===================
os.makedirs("/kaggle/working/pickle", exist_ok=True)

train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

clean_train_reviews, train_labels = [], []
for i, review in enumerate(train["review"]):
    clean_train_reviews.append(review_to_wordlist(review))
    train_labels.append(train["sentiment"][i])

clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review))

vocab = set(chain(*clean_train_reviews)) | set(chain(*clean_test_reviews))
vocab_size = len(vocab)

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    clean_train_reviews, train_labels, test_size=0.2, random_state=0)

# ===================【重点修改】适配Common Crawl 840B（glove-gensim分割逻辑） ===================
wvmodel = KeyedVectors(embed_size)
word_dict = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tokens = line.split()
        if len(tokens) <= embed_size:
            continue
        try:
            vec = np.array(tokens[-embed_size:], dtype=np.float32)
            word = " ".join(tokens[:-embed_size])
            word_dict[word] = vec
        except ValueError:
            continue
wvmodel.add_vectors(list(word_dict.keys()), list(word_dict.values()))
print(f"GloVe加载完成，载入词总数：{len(wvmodel)}")
# =========================================================================================

word_to_idx = {word: i + 1 for i, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0
idx_to_word = {i + 1: word for i, word in enumerate(vocab)}
idx_to_word[0] = '<unk>'

train_features = torch.tensor(pad_samples(encode_samples(train_reviews, word_to_idx)))
val_features = torch.tensor(pad_samples(encode_samples(val_reviews, word_to_idx)))
test_features = torch.tensor(pad_samples(encode_samples(clean_test_reviews, word_to_idx)))

train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 构建Embedding权重矩阵
weight = torch.zeros(vocab_size + 1, embed_size)
hit = 0
for word, idx in word_to_idx.items():
    if word in wvmodel:
        weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))
        hit += 1
print(f"词表匹配成功向量：{hit}/{len(word_to_idx)}")

pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
pickle.dump(
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab],
    open(pickle_file, 'wb'))
print('pickle文件生成完成！')

GloVe加载完成，载入词总数：2195895


/tmp/ipykernel_58/1117333043.py:114: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))


词表匹配成功向量：77554/101400
pickle文件生成完成！


In [9]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ===================== 路径常量（适配Kaggle） =====================
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/cnnlstm.csv"
# 【关键修复】提前创建输出文件夹，解决OSError
os.makedirs("/kaggle/working/result", exist_ok=True)

test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

num_epochs = 10
max_len = 512

embed_size = 300
num_filter = 128
filter_size = 3
pooling_size = 2

num_hiddens = 64
num_layers = 2

bidirectional = True
batch_size = 64
labels = 2
lr = 0.01   # 修复：原lr=0.8太大，无法训练
device = torch.device('cuda:0')
use_gpu = True

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_size, num_hiddens, num_layers, bidirectional, weight, labels, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.embed_size = embed_size
        self.num_filter = num_filter
        self.filter_size = filter_size
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.labels = labels
        self.use_gpu = use_gpu

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False

        self.conv1d = nn.Conv1d(self.embed_size, self.num_filter, self.filter_size, padding=1)
        self.activate = F.relu

        self.encoder = nn.LSTM(input_size=self.num_filter,
                                hidden_size=self.num_hiddens,
                                num_layers=self.num_layers,
                                bidirectional=self.bidirectional,
                                dropout=0)

        if self.bidirectional:
            self.decoder = nn.Linear(num_hiddens * 4, labels)
        else:
            self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)  # [batch, seq_len, embed_dim]

        convolution = self.activate(self.conv1d(embeddings.permute([0, 2, 1])))  # [batch, filter, seq]
        pooling = F.max_pool1d(convolution, kernel_size=pooling_size)  # [batch, filter, seq//2]

        lstm_input = pooling.permute([2, 0, 1])
        states, _ = self.encoder(lstm_input)

        encoding = torch.cat([states[0], states[-1]], dim=1)
        outputs = self.decoder(encoding)
        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    logging.info('loading data...')
    # 注意：确认你的pickle路径！如果文件直接在working，改成 pickle_file = "imdb_glove.pickle3"
    pickle_file = os.path.join('/kaggle/working/pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size, num_filter=num_filter, filter_size=filter_size,
                       num_hiddens=num_hiddens, num_layers=num_layers, bidirectional=bidirectional,
                       weight=weight, labels=labels, use_gpu=use_gpu)
    net.to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0, 0
        train_acc, val_acc = 0, 0
        n, m = 0, 0
        with tqdm(total=len(train_iter), desc='Epoch %d' % epoch) as pbar:
            for feature, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                optimizer.step()
                train_acc += accuracy_score(torch.argmax(score.cpu().data, dim=1), label.cpu())
                train_loss += loss.item() # 修复：去掉 .data，消除pytorch警告

                pbar.set_postfix({
                    'epoch': '%d' % epoch,
                    'train loss': '%.4f' % (train_loss / n),
                    'train acc': '%.2f' % (train_acc / n)
                })
                pbar.update(1)

            with torch.no_grad():
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label)
                    val_acc += accuracy_score(torch.argmax(val_score.cpu().data, dim=1), val_label.cpu())
                    val_losses += val_loss.item()
            end = time.time()
            runtime = end - start
            pbar.set_postfix({
                'epoch': '%d' % epoch,
                'train loss': '%.4f' % (train_loss / n),
                'train acc': '%.2f' % (train_acc / n),
                'val loss': '%.4f' % (val_losses / m),
                'val acc': '%.2f' % (val_acc / m),
                'time': '%.2f' % runtime
            })

    test_pred = []
    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv(RESULT_PATH, index=False, quoting=3)
    logging.info('result saved!')


2026-08-15 08:13:20,156: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-26c5996f-62fe-4a82-a6ca-bb4ba3b6de96.json
2026-08-15 08:13:20,157: INFO: loading data...
2026-08-15 08:13:20,697: INFO: data loaded!
Prediction: 100%|██████████| 391/391 [00:02<00:00, 158.61it/s]
2026-08-15 08:14:31,750: INFO: result saved!


In [13]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ===================== 路径配置 =====================
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/cnnlstm.csv"
BEST_MODEL_PATH = "/kaggle/working/result/best_model.pth"
os.makedirs("/kaggle/working/result", exist_ok=True)

test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

# 超参
num_epochs = 15
max_len = 512
embed_size = 300
num_filter = 128
filter_sizes = [3, 4, 5]
pooling_size = 2
num_hiddens = 128
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
dropout_rate = 0.3
grad_clip = 5.0
patience = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_sizes, num_hiddens, num_layers, bidirectional, weight, labels, dropout=0.3, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = True
        self.drop_emb = nn.Dropout(dropout)

        # 多尺度卷积：只用于全局特征
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_size, num_filter, fs, padding=fs // 2)
            for fs in filter_sizes
        ])

        cnn_global_dim = num_filter * len(filter_sizes)

        # 单尺度卷积：用于 LSTM 序列输入
        self.conv_for_lstm = nn.Conv1d(
            embed_size,
            num_filter,
            kernel_size=3,
            padding=1
        )

        lstm_in_dim = num_filter

        self.encoder = nn.LSTM(
            input_size=lstm_in_dim,
            hidden_size=num_hiddens,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0
        )

        lstm_out_dim = num_hiddens * 2 if bidirectional else num_hiddens

        self.dropout = nn.Dropout(dropout)

        # 融合 LSTM 首尾特征 + 多尺度 CNN 全局特征
        self.decoder = nn.Linear(lstm_out_dim * 2 + cnn_global_dim, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = self.drop_emb(embeddings)

        # [B, E, L]
        emb_t = embeddings.permute(0, 2, 1)

        # ---------- 多尺度 CNN 全局特征 ----------
        cnn_pool_outs = []
        for conv in self.convs:
            out = F.relu(conv(emb_t))
            pool_out = F.max_pool1d(out, kernel_size=out.size(-1)).squeeze(-1)
            cnn_pool_outs.append(pool_out)

        cnn_global_pool = torch.cat(cnn_pool_outs, dim=1)

        # ---------- 单尺度 CNN 序列特征，送入 LSTM ----------
        conv_seq = F.relu(self.conv_for_lstm(emb_t))

        pooling = F.max_pool1d(conv_seq, kernel_size=pooling_size)

        lstm_in = pooling.permute(2, 0, 1)

        states, _ = self.encoder(lstm_in)

        lstm_feature = torch.cat([states[0], states[-1]], dim=1)

        fusion = torch.cat([lstm_feature, cnn_global_pool], dim=1)
        fusion = self.dropout(fusion)

        out = self.decoder(fusion)

        return out

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {''.join(sys.argv)}")

    logger.info('loading data...')

    pickle_file = os.path.join('/kaggle/working/pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(open(pickle_file, 'rb'))

    logger.info('data loaded!')

    net = SentimentNet(
        embed_size=embed_size,
        num_filter=num_filter,
        filter_sizes=filter_sizes,
        num_hiddens=num_hiddens,
        num_layers=num_layers,
        bidirectional=bidirectional,
        weight=weight,
        labels=labels,
        dropout=dropout_rate
    )

    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features,)

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_acc = 0.0
    stop_cnt = 0

    for epoch in range(num_epochs):
        start = time.time()

        net.train()
        total_loss = 0.0
        tr_preds, tr_labels = [], []

        for feature, label in tqdm(train_iter, desc=f"Epoch {epoch} Train"):
            feature, label = feature.to(device), label.to(device)

            score = net(feature)
            loss = loss_function(score, label)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            optimizer.step()

            total_loss += loss.item()

            tr_preds.extend(torch.argmax(score.cpu(), dim=1).tolist())
            tr_labels.extend(label.cpu().tolist())

        train_acc = accuracy_score(tr_labels, tr_preds)

        net.eval()
        val_loss = 0.0
        va_preds, va_labels = [], []

        with torch.no_grad():
            for feat, lab in val_iter:
                feat, lab = feat.to(device), lab.to(device)
                s = net(feat)
                val_loss += loss_function(s, lab).item()
                va_preds.extend(torch.argmax(s.cpu(), dim=1).tolist())
                va_labels.extend(lab.cpu().tolist())

        val_acc = accuracy_score(va_labels, va_preds)
        scheduler.step(val_acc)

        logger.info(
            f"Epoch {epoch} | "
            f"Train Loss:{total_loss/len(train_iter):.4f} Acc:{train_acc:.4f} | "
            f"Val Loss:{val_loss/len(val_iter):.4f} Acc:{val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            stop_cnt = 0
            torch.save(net.state_dict(), BEST_MODEL_PATH)
            logger.info(f"Save Best Model, Best Val Acc:{best_acc:.4f}")
        else:
            stop_cnt += 1
            if stop_cnt >= patience:
                logger.info("Early Stop!")
                break

    net.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    net.eval()

    test_pred = []

    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            test_pred.extend(torch.argmax(test_score.cpu(), dim=1).numpy().tolist())

    result_output = pd.DataFrame({"id": test["id"], "sentiment": test_pred})
    result_output.to_csv(RESULT_PATH, index=False, quoting=3)

    logger.info("result saved!")


2026-08-15 08:21:22,380: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-26c5996f-62fe-4a82-a6ca-bb4ba3b6de96.json
2026-08-15 08:21:22,381: INFO: loading data...
2026-08-15 08:21:22,895: INFO: data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:33<00:00,  9.44it/s]
2026-08-15 08:21:58,279: INFO: Epoch 0 | Train Loss:0.4145 Acc:0.8044 | Val Loss:0.2763 Acc:0.8880
2026-08-15 08:21:58,506: INFO: Save Best Model, Best Val Acc:0.8880
Epoch 1 Train: 100%|██████████| 313/313 [00:31<00:00, 10.00it/s]
2026-08-15 08:22:31,918: INFO: Epoch 1 | Train Loss:0.2506 Acc:0.8990 | Val Loss:0.2515 Acc:0.8988
2026-08-15 08:22:32,256: INFO: Save Best Model, Best Val Acc:0.8988
Epoch 2 Train: 100%|██████████| 313/313 [00:31<00:00,  9.93it/s]
2026-08-15 08:23:05,982: INFO: Epoch 2 | Train Loss:0.1593 Acc:0.9404 | Val Loss:0.2475 Acc:0.9094
2026-08-15 08:23:06,319: INFO: Save Best Model, Best Val Acc:0.9094
Epoch 3 Train: 100%|████

In [14]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ===================== 路径配置 =====================
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/cnnlstm_att.csv"
BEST_MODEL_PATH = "/kaggle/working/result/best_model.pth"
os.makedirs("/kaggle/working/result", exist_ok=True)

test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

# 超参微调
num_epochs = 15
max_len = 512
embed_size = 300
num_filter = 128
filter_sizes = [3, 4, 5]
pooling_size = 2
num_hiddens = 128
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 8e-4
dropout_rate = 0.35
grad_clip = 5.0
patience = 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 自注意力模块
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_output):
        # lstm_output: [seq_len, batch, hidden]
        attn_weights = torch.tanh(self.attn(lstm_output))
        score = self.v(attn_weights).squeeze(-1)    # [seq_len, batch]
        alpha = F.softmax(score, dim=0)
        weighted = lstm_output * alpha.unsqueeze(-1)
        attn_out = torch.sum(weighted, dim=0)       # [batch, hidden]
        return attn_out

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_sizes, num_hiddens, num_layers, bidirectional, weight, labels, dropout=0.3, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = True
        self.drop_emb = nn.Dropout(dropout * 0.6)

        # 多尺度卷积：全局特征
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_size, num_filter, fs, padding=fs // 2)
            for fs in filter_sizes
        ])
        cnn_global_dim = num_filter * len(filter_sizes)

        # 单尺度卷积送入LSTM
        self.conv_for_lstm = nn.Conv1d(embed_size, num_filter, kernel_size=3, padding=1)
        lstm_in_dim = num_filter

        self.encoder = nn.LSTM(
            input_size=lstm_in_dim,
            hidden_size=num_hiddens,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0
        )
        lstm_out_dim = num_hiddens * 2 if bidirectional else num_hiddens

        # 注意力层
        self.attention = Attention(lstm_out_dim)

        self.dropout = nn.Dropout(dropout)
        # 特征融合：注意力输出 + LSTM首尾特征 + CNN多尺度全局特征
        self.decoder = nn.Linear(lstm_out_dim * 3 + cnn_global_dim, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = self.drop_emb(embeddings)
        emb_t = embeddings.permute(0, 2, 1)  # [B, E, L]

        # ----------多尺度CNN全局特征----------
        cnn_pool_outs = []
        for conv in self.convs:
            out = F.relu(conv(emb_t))
            pool_out = F.max_pool1d(out, kernel_size=out.size(-1)).squeeze(-1)
            cnn_pool_outs.append(pool_out)
        cnn_global_pool = torch.cat(cnn_pool_outs, dim=1)

        # ----------LSTM时序分支----------
        conv_seq = F.relu(self.conv_for_lstm(emb_t))
        pooling = F.max_pool1d(conv_seq, kernel_size=pooling_size)
        lstm_in = pooling.permute(2, 0, 1)  # [seq, B, C]

        states, (h_n, _) = self.encoder(lstm_in)
        # 注意力加权特征
        attn_feature = self.attention(states)
        # 拼接前向、后向最后一层hidden
        last_hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)
        # 全局平均时序特征
        mean_seq = torch.mean(states, dim=0)

        # 全部特征融合
        fusion = torch.cat([attn_feature, last_hidden, mean_seq, cnn_global_pool], dim=1)
        fusion = self.dropout(fusion)
        out = self.decoder(fusion)
        return out

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {''.join(sys.argv)}")

    logger.info('loading data...')
    pickle_file = os.path.join('/kaggle/working/pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(open(pickle_file, 'rb'))
    logger.info('data loaded!')

    net = SentimentNet(
        embed_size=embed_size,
        num_filter=num_filter,
        filter_sizes=filter_sizes,
        num_hiddens=num_hiddens,
        num_layers=num_layers,
        bidirectional=bidirectional,
        weight=weight,
        labels=labels,
        dropout=dropout_rate
    )
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.6, patience=1)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features,)

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_acc = 0.0
    stop_cnt = 0

    for epoch in range(num_epochs):
        start = time.time()
        net.train()
        total_loss = 0.0
        tr_preds, tr_labels = [], []

        for feature, label in tqdm(train_iter, desc=f"Epoch {epoch} Train"):
            feature, label = feature.to(device), label.to(device)
            score = net(feature)
            loss = loss_function(score, label)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            optimizer.step()

            total_loss += loss.item()
            tr_preds.extend(torch.argmax(score.cpu(), dim=1).tolist())
            tr_labels.extend(label.cpu().tolist())

        train_acc = accuracy_score(tr_labels, tr_preds)

        net.eval()
        val_loss = 0.0
        va_preds, va_labels = [], []
        with torch.no_grad():
            for feat, lab in val_iter:
                feat, lab = feat.to(device), lab.to(device)
                s = net(feat)
                val_loss += loss_function(s, lab).item()
                va_preds.extend(torch.argmax(s.cpu(), dim=1).tolist())
                va_labels.extend(lab.cpu().tolist())
        val_acc = accuracy_score(va_labels, va_preds)
        scheduler.step(val_acc)

        logger.info(
            f"Epoch {epoch} | "
            f"Train Loss:{total_loss/len(train_iter):.4f} Acc:{train_acc:.4f} | "
            f"Val Loss:{val_loss/len(val_iter):.4f} Acc:{val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            stop_cnt = 0
            torch.save(net.state_dict(), BEST_MODEL_PATH)
            logger.info(f"Save Best Model, Best Val Acc:{best_acc:.4f}")
        else:
            stop_cnt += 1
            if stop_cnt >= patience:
                logger.info("Early Stop!")
                break

    net.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    net.eval()
    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            test_pred.extend(torch.argmax(test_score.cpu(), dim=1).numpy().tolist())

    result_output = pd.DataFrame({"id": test["id"], "sentiment": test_pred})
    result_output.to_csv(RESULT_PATH, index=False, quoting=3)
    logger.info("result saved!")


2026-08-15 08:27:12,073: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-26c5996f-62fe-4a82-a6ca-bb4ba3b6de96.json
2026-08-15 08:27:12,074: INFO: loading data...
2026-08-15 08:27:12,615: INFO: data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:33<00:00,  9.25it/s]
2026-08-15 08:27:48,814: INFO: Epoch 0 | Train Loss:0.4051 Acc:0.8041 | Val Loss:0.2772 Acc:0.8914
2026-08-15 08:27:49,138: INFO: Save Best Model, Best Val Acc:0.8914
Epoch 1 Train: 100%|██████████| 313/313 [00:32<00:00,  9.65it/s]
2026-08-15 08:28:23,787: INFO: Epoch 1 | Train Loss:0.2226 Acc:0.9099 | Val Loss:0.2420 Acc:0.9076
2026-08-15 08:28:24,108: INFO: Save Best Model, Best Val Acc:0.9076
Epoch 2 Train: 100%|██████████| 313/313 [00:32<00:00,  9.73it/s]
2026-08-15 08:28:58,543: INFO: Epoch 2 | Train Loss:0.1313 Acc:0.9515 | Val Loss:0.2554 Acc:0.9072
Epoch 3 Train: 100%|██████████| 313/313 [00:32<00:00,  9.52it/s]
2026-08-15 08:29:33,669: IN

In [15]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ===================== 路径配置 =====================
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/cnnlstm_multiatt.csv"
BEST_MODEL_PATH = "/kaggle/working/result/best_model.pth"
os.makedirs("/kaggle/working/result", exist_ok=True)

test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

# 超参微调
num_epochs = 15
max_len = 512
embed_size = 300
num_filter = 128
filter_sizes = [3, 4, 5]
pooling_size = 2
num_hiddens = 128
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 8e-4
dropout_rate = 0.35
grad_clip = 5.0
patience = 4
head_num = 4  # 多头注意力头数
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 多头注意力模块
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = hidden_dim // heads
        assert self.head_dim * heads == hidden_dim, "hidden_dim必须能被头数整除"
        
        self.w_q = nn.Linear(hidden_dim, hidden_dim)
        self.w_k = nn.Linear(hidden_dim, hidden_dim)
        self.w_v = nn.Linear(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, lstm_out):
        # lstm_out [seq_len, batch, hidden]
        seq_len, batch, _ = lstm_out.shape
        Q = self.w_q(lstm_out)
        K = self.w_k(lstm_out)
        V = self.w_v(lstm_out)

        # 切分多头
        Q = Q.view(seq_len, batch, self.heads, self.head_dim).permute(1,2,0,3)
        K = K.view(seq_len, batch, self.heads, self.head_dim).permute(1,2,0,3)
        V = V.view(seq_len, batch, self.heads, self.head_dim).permute(1,2,0,3)

        # 缩放点积注意力
        attn_score = torch.matmul(Q, K.transpose(-2,-1)) / (self.head_dim ** 0.5)
        attn_weight = F.softmax(attn_score, dim=-1)
        attn_weight = self.dropout(attn_weight)
        attn_out = torch.matmul(attn_weight, V)

        attn_out = attn_out.permute(2,0,1,3).contiguous()
        attn_out = attn_out.view(seq_len, batch, -1)
        attn_out = self.fc(attn_out)
        return attn_out

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_sizes, num_hiddens, num_layers, bidirectional, weight, heads, labels, dropout=0.3, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = True
        self.drop_emb = nn.Dropout(dropout * 0.6)

        # 多尺度卷积：全局特征
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_size, num_filter, fs, padding=fs // 2)
            for fs in filter_sizes
        ])
        cnn_global_dim = num_filter * len(filter_sizes)

        # 单尺度卷积送入LSTM
        self.conv_for_lstm = nn.Conv1d(embed_size, num_filter, kernel_size=3, padding=1)
        lstm_in_dim = num_filter

        self.encoder = nn.LSTM(
            input_size=lstm_in_dim,
            hidden_size=num_hiddens,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0
        )
        lstm_out_dim = num_hiddens * 2 if bidirectional else num_hiddens

        # 多头注意力 + Norm
        self.multi_attn = MultiHeadAttention(lstm_out_dim, heads)
        self.ln_attn = nn.LayerNorm(lstm_out_dim)

        self.dropout = nn.Dropout(dropout)
        # 特征维度：多头注意力输出 + LSTM首尾隐状态 + CNN全局特征
        self.decoder = nn.Linear(lstm_out_dim * 2 + cnn_global_dim, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = self.drop_emb(embeddings)
        emb_t = embeddings.permute(0, 2, 1)  # [B, E, L]

        # ----------多尺度CNN全局特征----------
        cnn_pool_outs = []
        for conv in self.convs:
            out = F.relu(conv(emb_t))
            pool_out = F.max_pool1d(out, kernel_size=out.size(-1)).squeeze(-1)
            cnn_pool_outs.append(pool_out)
        cnn_global_pool = torch.cat(cnn_pool_outs, dim=1)

        # ----------LSTM时序分支----------
        conv_seq = F.relu(self.conv_for_lstm(emb_t))
        pooling = F.max_pool1d(conv_seq, kernel_size=pooling_size)
        lstm_in = pooling.permute(2, 0, 1)  # [seq, B, C]

        states, (h_n, _) = self.encoder(lstm_in)
        # 多头注意力 + 残差
        attn_raw = self.multi_attn(states)
        attn_feature = self.ln_attn(states + attn_raw)  # 残差连接
        attn_feature = torch.mean(attn_feature, dim=0) # 时序维度平均

        # 拼接双向LSTM最后一层隐状态
        last_hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)

        # 特征融合
        fusion = torch.cat([attn_feature, last_hidden, cnn_global_pool], dim=1)
        fusion = self.dropout(fusion)
        out = self.decoder(fusion)
        return out

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {''.join(sys.argv)}")

    logger.info('loading data...')
    pickle_file = os.path.join('/kaggle/working/pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(open(pickle_file, 'rb'))
    logger.info('data loaded!')

    net = SentimentNet(
        embed_size=embed_size,
        num_filter=num_filter,
        filter_sizes=filter_sizes,
        num_hiddens=num_hiddens,
        num_layers=num_layers,
        bidirectional=bidirectional,
        weight=weight,
        heads=head_num,
        labels=labels,
        dropout=dropout_rate
    )
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    # ==========分层学习率关键优化！embedding使用更小lr============
    embedding_params = list(map(id, net.embedding.parameters()))
    base_params = filter(lambda p: id(p) not in embedding_params, net.parameters())
    optimizer = optim.AdamW([
        {'params': net.embedding.parameters(), 'lr': lr * 0.3},
        {'params': base_params, 'lr': lr}
    ], weight_decay=1e-4)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.6, patience=1)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features,)

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_acc = 0.0
    stop_cnt = 0

    for epoch in range(num_epochs):
        start = time.time()
        net.train()
        total_loss = 0.0
        tr_preds, tr_labels = [], []

        for feature, label in tqdm(train_iter, desc=f"Epoch {epoch} Train"):
            feature, label = feature.to(device), label.to(device)
            score = net(feature)
            loss = loss_function(score, label)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            optimizer.step()

            total_loss += loss.item()
            tr_preds.extend(torch.argmax(score.cpu(), dim=1).tolist())
            tr_labels.extend(label.cpu().tolist())

        train_acc = accuracy_score(tr_labels, tr_preds)

        net.eval()
        val_loss = 0.0
        va_preds, va_labels = [], []
        with torch.no_grad():
            for feat, lab in val_iter:
                feat, lab = feat.to(device), lab.to(device)
                s = net(feat)
                val_loss += loss_function(s, lab).item()
                va_preds.extend(torch.argmax(s.cpu(), dim=1).tolist())
                va_labels.extend(lab.cpu().tolist())
        val_acc = accuracy_score(va_labels, va_preds)
        scheduler.step(val_acc)

        logger.info(
            f"Epoch {epoch} | "
            f"Train Loss:{total_loss/len(train_iter):.4f} Acc:{train_acc:.4f} | "
            f"Val Loss:{val_loss/len(val_iter):.4f} Acc:{val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            stop_cnt = 0
            torch.save(net.state_dict(), BEST_MODEL_PATH)
            logger.info(f"Save Best Model, Best Val Acc:{best_acc:.4f}")
        else:
            stop_cnt += 1
            if stop_cnt >= patience:
                logger.info("Early Stop!")
                break

    net.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    net.eval()
    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            test_pred.extend(torch.argmax(test_score.cpu(), dim=1).numpy().tolist())

    result_output = pd.DataFrame({"id": test["id"], "sentiment": test_pred})
    result_output.to_csv(RESULT_PATH, index=False, quoting=3)
    logger.info("result saved!")


2026-08-15 08:34:40,811: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-26c5996f-62fe-4a82-a6ca-bb4ba3b6de96.json
2026-08-15 08:34:40,812: INFO: loading data...
2026-08-15 08:34:41,362: INFO: data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:39<00:00,  7.84it/s]
2026-08-15 08:35:24,012: INFO: Epoch 0 | Train Loss:0.4405 Acc:0.7871 | Val Loss:0.3215 Acc:0.8652
2026-08-15 08:35:24,361: INFO: Save Best Model, Best Val Acc:0.8652
Epoch 1 Train: 100%|██████████| 313/313 [00:38<00:00,  8.23it/s]
2026-08-15 08:36:05,037: INFO: Epoch 1 | Train Loss:0.2681 Acc:0.8901 | Val Loss:0.2543 Acc:0.8974
2026-08-15 08:36:05,362: INFO: Save Best Model, Best Val Acc:0.8974
Epoch 2 Train: 100%|██████████| 313/313 [00:38<00:00,  8.11it/s]
2026-08-15 08:36:46,627: INFO: Epoch 2 | Train Loss:0.2105 Acc:0.9155 | Val Loss:0.2702 Acc:0.8896
Epoch 3 Train: 100%|██████████| 313/313 [00:38<00:00,  8.15it/s]
2026-08-15 08:37:27,662: IN

In [16]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ===================== 路径配置 =====================
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/cnnlstm_final.csv"
BEST_MODEL_PATH = "/kaggle/working/result/best_model.pth"
os.makedirs("/kaggle/working/result", exist_ok=True)

test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

# 超参微调
num_epochs = 15
max_len = 512
embed_size = 300
num_filter = 128
filter_sizes = [3, 4, 5]
pooling_size = 2
num_hiddens = 128
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 8e-4
dropout_rate = 0.35
grad_clip = 5.0
patience = 4
head_num = 4
warmup_epoch = 2  # 前2轮学习率预热
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 多头注意力模块
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = hidden_dim // heads
        assert self.head_dim * heads == hidden_dim, "hidden_dim必须能被头数整除"

        self.w_q = nn.Linear(hidden_dim, hidden_dim)
        self.w_k = nn.Linear(hidden_dim, hidden_dim)
        self.w_v = nn.Linear(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, lstm_out):
        seq_len, batch, _ = lstm_out.shape
        Q = self.w_q(lstm_out)
        K = self.w_k(lstm_out)
        V = self.w_v(lstm_out)

        Q = Q.view(seq_len, batch, self.heads, self.head_dim).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch, self.heads, self.head_dim).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch, self.heads, self.head_dim).permute(1, 2, 0, 3)

        attn_score = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weight = F.softmax(attn_score, dim=-1)
        attn_weight = self.dropout(attn_weight)
        attn_out = torch.matmul(attn_weight, V)

        attn_out = attn_out.permute(2, 0, 1, 3).contiguous()
        attn_out = attn_out.view(seq_len, batch, -1)
        attn_out = self.fc(attn_out)
        return attn_out

# 残差卷积块
class ResConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, padding=kernel//2)
        self.ln = nn.LayerNorm(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=kernel//2)
        self.shortcut = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.drop = nn.Dropout(0.25)

    def forward(self, x):
        residual = self.shortcut(x)
        x = F.relu(self.conv1(x))
        x = self.drop(x)
        x = self.conv2(x)
        x = x + residual
        x = self.ln(x.transpose(1,2)).transpose(1,2)
        return F.relu(x)

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_sizes, num_hiddens, num_layers, bidirectional, weight, heads, labels, dropout=0.3, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = True
        self.drop_emb = nn.Dropout(dropout * 0.6)

        # 多尺度CNN全局分支
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_size, num_filter, fs, padding=fs // 2)
            for fs in filter_sizes
        ])
        cnn_global_dim = num_filter * len(filter_sizes)

        # LSTM时序分支 + 残差卷积预处理
        self.conv_for_lstm = ResConv1d(embed_size, num_filter, kernel=3)
        lstm_in_dim = num_filter

        self.encoder = nn.LSTM(
            input_size=lstm_in_dim,
            hidden_size=num_hiddens,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0
        )
        lstm_out_dim = num_hiddens * 2 if bidirectional else num_hiddens

        self.multi_attn = MultiHeadAttention(lstm_out_dim, heads)
        self.ln_attn = nn.LayerNorm(lstm_out_dim)

        self.dropout = nn.Dropout(dropout)
        fusion_dim = lstm_out_dim * 3 + cnn_global_dim
        # 多层融合头，替代单层Linear
        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.LayerNorm(fusion_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim // 2, labels)
        )

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = self.drop_emb(embeddings)
        emb_t = embeddings.permute(0, 2, 1)  # [B, E, L]

        # ----------多尺度CNN全局特征----------
        cnn_pool_outs = []
        for conv in self.convs:
            out = F.relu(conv(emb_t))
            pool_out = F.max_pool1d(out, kernel_size=out.size(-1)).squeeze(-1)
            cnn_pool_outs.append(pool_out)
        cnn_global_pool = torch.cat(cnn_pool_outs, dim=1)

        # ----------LSTM时序分支----------
        conv_seq = self.conv_for_lstm(emb_t)
        pooling = F.max_pool1d(conv_seq, kernel_size=pooling_size)
        lstm_in = pooling.permute(2, 0, 1)  # [seq, B, C]

        states, (h_n, _) = self.encoder(lstm_in)
        # 多头注意力 + 残差
        attn_raw = self.multi_attn(states)
        attn_seq = self.ln_attn(states + attn_raw)

        attn_feature = torch.mean(attn_seq, dim=0)
        max_seq_feature = torch.max(attn_seq, dim=0)[0]
        last_hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)

        # 融合：注意力均值 + 时序最大值 + LSTM双向末态 + CNN全局特征
        fusion = torch.cat([attn_feature, max_seq_feature, last_hidden, cnn_global_pool], dim=1)
        fusion = self.dropout(fusion)
        out = self.mlp(fusion)
        return out

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {''.join(sys.argv)}")

    logger.info('loading data...')
    pickle_file = os.path.join('/kaggle/working/pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(open(pickle_file, 'rb'))
    logger.info('data loaded!')

    net = SentimentNet(
        embed_size=embed_size,
        num_filter=num_filter,
        filter_sizes=filter_sizes,
        num_hiddens=num_hiddens,
        num_layers=num_layers,
        bidirectional=bidirectional,
        weight=weight,
        heads=head_num,
        labels=labels,
        dropout=dropout_rate
    )
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    embedding_params = list(map(id, net.embedding.parameters()))
    base_params = filter(lambda p: id(p) not in embedding_params, net.parameters())
    optimizer = optim.AdamW([
        {'params': net.embedding.parameters(), 'lr': lr * 0.3},
        {'params': base_params, 'lr': lr}
    ], weight_decay=1e-4)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.6, patience=1)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features,)

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_acc = 0.0
    stop_cnt = 0

    for epoch in range(num_epochs):
        # 学习率预热
        if epoch < warmup_epoch:
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr * 0.2 + (lr * 0.8) * (epoch / warmup_epoch)

        start = time.time()
        net.train()
        total_loss = 0.0
        tr_preds, tr_labels = [], []

        for feature, label in tqdm(train_iter, desc=f"Epoch {epoch} Train"):
            feature, label = feature.to(device), label.to(device)
            score = net(feature)
            loss = loss_function(score, label)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            optimizer.step()

            total_loss += loss.item()
            tr_preds.extend(torch.argmax(score.cpu(), dim=1).tolist())
            tr_labels.extend(label.cpu().tolist())

        train_acc = accuracy_score(tr_labels, tr_preds)

        net.eval()
        val_loss = 0.0
        va_preds, va_labels = [], []
        with torch.no_grad():
            for feat, lab in val_iter:
                feat, lab = feat.to(device), lab.to(device)
                s = net(feat)
                val_loss += loss_function(s, lab).item()
                va_preds.extend(torch.argmax(s.cpu(), dim=1).tolist())
                va_labels.extend(lab.cpu().tolist())
        val_acc = accuracy_score(va_labels, va_preds)
        scheduler.step(val_acc)

        logger.info(
            f"Epoch {epoch} | "
            f"Train Loss:{total_loss/len(train_iter):.4f} Acc:{train_acc:.4f} | "
            f"Val Loss:{val_loss/len(val_iter):.4f} Acc:{val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            stop_cnt = 0
            torch.save(net.state_dict(), BEST_MODEL_PATH)
            logger.info(f"Save Best Model, Best Val Acc:{best_acc:.4f}")
        else:
            stop_cnt += 1
            if stop_cnt >= patience:
                logger.info("Early Stop!")
                break

    net.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    net.eval()
    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            test_pred.extend(torch.argmax(test_score.cpu(), dim=1).numpy().tolist())

    result_output = pd.DataFrame({"id": test["id"], "sentiment": test_pred})
    result_output.to_csv(RESULT_PATH, index=False, quoting=3)
    logger.info("result saved!")


2026-08-15 08:43:41,329: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-26c5996f-62fe-4a82-a6ca-bb4ba3b6de96.json
2026-08-15 08:43:41,330: INFO: loading data...
2026-08-15 08:43:41,841: INFO: data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:43<00:00,  7.13it/s]
2026-08-15 08:44:28,813: INFO: Epoch 0 | Train Loss:0.5638 Acc:0.6748 | Val Loss:0.3316 Acc:0.8590
2026-08-15 08:44:29,145: INFO: Save Best Model, Best Val Acc:0.8590
Epoch 1 Train: 100%|██████████| 313/313 [00:41<00:00,  7.52it/s]
2026-08-15 08:45:13,695: INFO: Epoch 1 | Train Loss:0.3339 Acc:0.8552 | Val Loss:0.3114 Acc:0.8696
2026-08-15 08:45:14,026: INFO: Save Best Model, Best Val Acc:0.8696
Epoch 2 Train: 100%|██████████| 313/313 [00:42<00:00,  7.31it/s]
2026-08-15 08:45:59,826: INFO: Epoch 2 | Train Loss:0.2167 Acc:0.9171 | Val Loss:0.2277 Acc:0.9092
2026-08-15 08:46:00,151: INFO: Save Best Model, Best Val Acc:0.9092
Epoch 3 Train: 100%|████